# PART B — Instruction Fine-Tuning with QLoRA  

## Part B1 — Instruction Dataset Creation  

Instruction pairs are generated using code.     
It is shared as seperate .py file as it is quite large.    
The generation code itself was generated using AI (GPT-sol)

```
data/cleaned - contains all the txt files used for CPT training in PART-A
```

In [10]:
import sys
import os
from pathlib import Path

sys.path.append(os.path.abspath('code'))
from create_instruction_dataset import create_instruction_dataset
report = create_instruction_dataset(
            "data/cleaned",
            "data/instructions/new",
            total_pairs=100,
            train_ratio=0.8,
            max_pairs_per_document=10,
            seed=43,
        )
train = report["train"]
evaluation = report["evaluation"]
print("=== Instruction Dataset Summary ===")
print(f"Input text files:       {report['input_text_file_count']:,}")
print(f"Usable unique pairs:    {report['candidate_pair_count_after_deduplication']:,}")
print(f"Training pairs:         {train['count']:,}")
print(f"Evaluation pairs:       {evaluation['count']:,}")
print(f"Source overlap:         {report['source_overlap_count']}")
print(f"Grounding check:        {report['grounding_check']}")

=== Instruction Dataset Summary ===
Input text files:       202
Usable unique pairs:    810
Training pairs:         80
Evaluation pairs:       20
Source overlap:         0
Grounding check:        passed


In [14]:
!tail -1 "data/instructions/new/evaluation.jsonl"

{"instruction": "What Ethernet interface scale can Cisco N9500 Series modular switches provide?", "response": "The Cisco N9500 Series modular switches support a comprehensive selection of line cards and fabric modules that provide 1-, 10-, 25-, 40-, 50-, 100-, 200-, and 400-Gigabit Ethernet interfaces. Using these line cards the Cisco N9500 Series switches can be configured with up to. 1. 256 400-Gigabit Ethernet ports (or) 2. 524 200-Gigabit Ethernet ports# (or) 3. 1024 100-Gigabit Ethernet ports (or) 4. 2048 50-Gigabit Ethernet ports (or) 5. 1024 40-Gigabit Ethernet ports (or) 6. 2304 25-Gigabit Ethernet ports (or) 7. 2304 1/10-Gigabit Ethernet ports"}


In [16]:
!head -1 "data/instructions/new/train.jsonl"

{"instruction": "How do you configure BFD on EIGRP?", "response": "You can configure BFD for the Enhanced Interior Gateway Routing Protocol (EIGRP). Before you begin Enable the BFD feature. See the Enabling the BFD Feature section. Configure the BFD session parameters. See the Configuring Global BFD Parameters section or the Configuring BFD on an Interface section. Enable the EIGRP feature. See the Cisco Nexus 9000 Series NX-OS Unicast Routing Configuration Guide for more information."}


## Part B2 — QLoRA Fine-Tuning with 3 Adapter Configurations  

### Imports and GPU Settings

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch

print("Visible GPU count:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0))

assert torch.cuda.device_count() == 1


Visible GPU count: 1
GPU: NVIDIA A100-SXM4-80GB


In [3]:
from trl import SFTConfig, SFTTrainer

In [2]:
import math
import os
from pathlib import Path

import torch
from datasets import load_dataset
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from trl import SFTConfig, SFTTrainer

### QLoRA Fine Tuning Code

In [4]:

def run_qlora_finetuning_with_chat_template(
    # Model and data
    model_name="Qwen/Qwen2.5-7B",
    train_file="1a/data/instructions/train.jsonl",
    evaluation_file="1a/data/instructions/evaluation.jsonl",
    output_dir="1a/output/qwen2.5-7b/adapter-r8",
    cache_dir=None,

    # LoRA adapter
    lora_rank=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules="all-linear",
    lora_bias="none",

    # QLoRA quantization
    quantization_type="nf4",
    use_double_quantization=True,

    # Training
    epochs=3,
    learning_rate=1e-4,
    train_batch_size=1,
    evaluation_batch_size=1,
    gradient_accumulation_steps=8,
    max_length=1024,
    warmup_ratio=0.03,
    weight_decay=0.0,
    max_grad_norm=1.0,
    optimizer="paged_adamw_8bit",
    lr_scheduler_type="linear",

    # Logging and checkpoints
    logging_steps=5,
    disable_tqdm=True,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    save_total_limit=2,

    # Memory and reproducibility
    gradient_checkpointing=True,
    packing=False,
    seed=42,
    trust_remote_code=False,
    resume_from_checkpoint=None,
):
    """
    Fine-tune a causal language model using 4-bit QLoRA.

    Expected JSONL dataset format:
        {"instruction": "...", "response": "..."}

    Each instruction/response pair is formatted with the tokenizer's
    model-specific chat template before tokenization.

    Only assistant-response tokens contribute to training loss.
    Instruction and padding tokens receive labels of -100.
    """

    # ---------------------------------------------------------
    # Validate runtime and arguments
    # ---------------------------------------------------------

    if not torch.cuda.is_available():
        raise RuntimeError(
            "QLoRA training requires a CUDA-capable GPU."
        )

    if torch.cuda.device_count() != 1:
        raise RuntimeError(
            f"{torch.cuda.device_count()} GPUs are visible. "
            "This notebook configuration expects exactly one visible GPU. "
            "Set CUDA_VISIBLE_DEVICES=0 before importing torch, then "
            "restart the kernel."
        )

    if packing:
        raise ValueError(
            "packing=True is not supported because this implementation "
            "constructs explicit response-only labels."
        )

    if max_length <= 0:
        raise ValueError("max_length must be greater than zero.")

    if lora_rank <= 0:
        raise ValueError("lora_rank must be greater than zero.")

    train_file = Path(train_file).expanduser().resolve()
    evaluation_file = Path(evaluation_file).expanduser().resolve()
    output_dir = Path(output_dir).expanduser().resolve()

    if not train_file.is_file():
        raise FileNotFoundError(
            f"Training dataset does not exist: {train_file}"
        )

    if not evaluation_file.is_file():
        raise FileNotFoundError(
            f"Evaluation dataset does not exist: {evaluation_file}"
        )

    output_dir.mkdir(parents=True, exist_ok=True)

    if cache_dir is None:
        cache_dir = os.getenv("CACHE_DIR")

    if cache_dir:
        cache_dir = str(Path(cache_dir).expanduser().resolve())

    set_seed(seed)

    compute_dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )

    gpu_major_version = torch.cuda.get_device_capability(0)[0]
    use_tf32 = gpu_major_version >= 8

    print("=" * 60)
    print("QLoRA TRAINING CONFIGURATION")
    print("=" * 60)
    print(f"Model/CPT checkpoint:         {model_name}")
    print(f"Training data:                {train_file}")
    print(f"Evaluation data:              {evaluation_file}")
    print(f"Output directory:             {output_dir}")
    print(f"GPU:                          {torch.cuda.get_device_name(0)}")
    print(f"Compute dtype:                {compute_dtype}")
    print(f"TF32 enabled:                 {use_tf32}")
    print(f"LoRA rank:                    {lora_rank}")
    print(f"LoRA alpha:                   {lora_alpha}")
    print(f"LoRA scaling:                 {lora_alpha / lora_rank:.2f}")
    print(f"LoRA dropout:                 {lora_dropout}")
    print(f"Target modules:               {target_modules}")
    print(f"Epochs:                       {epochs}")
    print(f"Learning rate:                {learning_rate}")
    print(f"Micro-batch size:             {train_batch_size}")
    print(f"Gradient accumulation:        {gradient_accumulation_steps}")
    print(
        "Effective batch size:          "
        f"{train_batch_size * gradient_accumulation_steps}"
    )
    print(f"Maximum sequence length:      {max_length}")
    print(f"Progress bar disabled:        {disable_tqdm}")
    print("=" * 60)

    # ---------------------------------------------------------
    # Load tokenizer
    # ---------------------------------------------------------

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        cache_dir=cache_dir,
        trust_remote_code=trust_remote_code,
        use_fast=True,
    )

    if tokenizer.eos_token is None:
        raise ValueError(
            "The selected tokenizer does not define an EOS token."
        )

    # This is safe with our custom collator because padding labels
    # are explicitly set to -100, while genuine EOS labels remain.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    chat_template = getattr(tokenizer, "chat_template", None)

    if not chat_template:
        raise ValueError(
            "The selected tokenizer does not define a chat template. "
            "Assignment 1A requires SFT data to use the model's chat "
            "template. Use a compatible chat/instruct tokenizer or save "
            "an explicit chat template with the CPT checkpoint before "
            "running QLoRA."
        )

    # This smoke test validates not only that chat_template is populated,
    # but also that this tokenizer can resolve and execute the template.
    try:
        tokenizer.apply_chat_template(
            [{"role": "user", "content": "Template validation"}],
            tokenize=True,
            add_generation_prompt=True,
        )
    except Exception as error:
        raise ValueError(
            "The tokenizer has a chat template, but it could not be "
            "applied to user/assistant messages."
        ) from error

    print(
        f"EOS token: {tokenizer.eos_token!r} "
        f"(ID: {tokenizer.eos_token_id})"
    )
    print(
        f"PAD token: {tokenizer.pad_token!r} "
        f"(ID: {tokenizer.pad_token_id})"
    )
    print("Chat template: detected and validated")

    # ---------------------------------------------------------
    # Configure 4-bit base-model loading
    # ---------------------------------------------------------

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type=quantization_type,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=use_double_quantization,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        cache_dir=cache_dir,
        quantization_config=quantization_config,
        device_map={"": 0},
        torch_dtype=compute_dtype,
        trust_remote_code=trust_remote_code,
    )

    model.config.use_cache = False
    model.config.pad_token_id = tokenizer.pad_token_id

    # Prepare the quantized model for adapter training.
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=gradient_checkpointing,
    )

    # ---------------------------------------------------------
    # Attach LoRA adapter
    # ---------------------------------------------------------

    lora_config = LoraConfig(
        task_type="CAUSAL_LM",
        inference_mode=False,
        r=lora_rank,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=target_modules,
        bias=lora_bias,
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # ---------------------------------------------------------
    # Load JSONL datasets
    # ---------------------------------------------------------

    train_dataset = load_dataset(
        "json",
        data_files=str(train_file),
        split="train",
    )

    evaluation_dataset = load_dataset(
        "json",
        data_files=str(evaluation_file),
        split="train",
    )

    required_columns = {"instruction", "response"}

    for dataset_name, dataset in (
        ("training", train_dataset),
        ("evaluation", evaluation_dataset),
    ):
        missing_columns = (
            required_columns - set(dataset.column_names)
        )

        if missing_columns:
            raise ValueError(
                f"The {dataset_name} dataset is missing columns: "
                f"{sorted(missing_columns)}"
            )

        if len(dataset) == 0:
            raise ValueError(
                f"The {dataset_name} dataset contains no records."
            )

    print(f"Training examples:            {len(train_dataset):,}")
    print(f"Evaluation examples:          {len(evaluation_dataset):,}")

    # ---------------------------------------------------------
    # Format records
    # ---------------------------------------------------------

    def format_record(record):
        """
        Render the record with the tokenizer's model-specific chat
        template for inspection. Token IDs are created separately below.
        """
        instruction = record["instruction"].strip()
        response = record["response"].strip()

        if not instruction:
            raise ValueError("Encountered an empty instruction.")

        if not response:
            raise ValueError("Encountered an empty response.")

        prompt_messages = [
            {"role": "user", "content": instruction},
        ]
        full_messages = [
            *prompt_messages,
            {"role": "assistant", "content": response},
        ]

        return {
            "prompt_text": tokenizer.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=True,
            ),
            "text": tokenizer.apply_chat_template(
                full_messages,
                tokenize=False,
                add_generation_prompt=False,
            ),
        }

    train_dataset = train_dataset.map(format_record)
    evaluation_dataset = evaluation_dataset.map(format_record)

    print("\nFormatted training example:")
    print(train_dataset[0]["text"][:1000])

    # ---------------------------------------------------------
    # Explicit response-only tokenization
    # ---------------------------------------------------------

    def tokenize_record(record):
        """
        Construct causal-LM labels explicitly.

        Theory:
        A causal language model normally predicts every token in the
        sequence. For supervised instruction tuning, we only want the
        answer to contribute to loss.

        Implementation:
        - Prompt labels are set to -100.
        - Response labels contain their actual token IDs.
        - Chat-template assistant/end tokens remain in response labels.
        - Padding labels are later set to -100 by the collator.
        """
        instruction = record["instruction"].strip()
        response = record["response"].strip()

        prompt_messages = [
            {"role": "user", "content": instruction},
        ]
        full_messages = [
            *prompt_messages,
            {"role": "assistant", "content": response},
        ]

        # add_generation_prompt=True includes the model-specific assistant
        # prefix. It is context, not an answer, so its labels are masked.
        prompt_ids = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=True,
            add_generation_prompt=True,
        )
        full_ids = tokenizer.apply_chat_template(
            full_messages,
            tokenize=True,
            add_generation_prompt=False,
        )

        if full_ids[:len(prompt_ids)] != prompt_ids:
            raise ValueError(
                "The tokenizer's completed chat is not prefixed by its "
                "generation prompt, so the assistant-response boundary "
                "cannot be masked safely."
            )

        if len(prompt_ids) >= max_length:
            raise ValueError(
                "The instruction leaves no room for a response. "
                f"Prompt tokens: {len(prompt_ids)}, "
                f"max_length: {max_length}"
            )

        # Everything after the generation prompt is the assistant target,
        # including model-specific end-of-message control tokens.
        response_ids = full_ids[len(prompt_ids):]
        response_ids = response_ids[:max_length - len(prompt_ids)]

        if not response_ids:
            raise ValueError(
                "The chat template produced no assistant-response tokens."
            )

        input_ids = prompt_ids + response_ids

        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": (
                [-100] * len(prompt_ids)
                + response_ids.copy()
            ),
        }

    tokenized_train_dataset = train_dataset.map(
        tokenize_record,
        remove_columns=train_dataset.column_names,
    )

    tokenized_evaluation_dataset = evaluation_dataset.map(
        tokenize_record,
        remove_columns=evaluation_dataset.column_names,
    )

    # ---------------------------------------------------------
    # Custom padding collator
    # ---------------------------------------------------------

    def completion_collator(features):
        """
        Pad each batch dynamically.

        Input IDs use the tokenizer's pad token.
        Attention is zero on padding.
        Labels use -100 on padding so it is ignored by loss.
        """
        batch_max_length = max(
            len(feature["input_ids"])
            for feature in features
        )

        batch_input_ids = []
        batch_attention_masks = []
        batch_labels = []

        for feature in features:
            padding_length = (
                batch_max_length - len(feature["input_ids"])
            )

            batch_input_ids.append(
                feature["input_ids"]
                + [tokenizer.pad_token_id] * padding_length
            )

            batch_attention_masks.append(
                feature["attention_mask"]
                + [0] * padding_length
            )

            batch_labels.append(
                feature["labels"]
                + [-100] * padding_length
            )

        return {
            "input_ids": torch.tensor(
                batch_input_ids,
                dtype=torch.long,
            ),
            "attention_mask": torch.tensor(
                batch_attention_masks,
                dtype=torch.long,
            ),
            "labels": torch.tensor(
                batch_labels,
                dtype=torch.long,
            ),
        }

    # ---------------------------------------------------------
    # Verify response-only labels
    # ---------------------------------------------------------

    sample = tokenized_train_dataset[0]

    trained_token_ids = [
        token_id
        for token_id in sample["labels"]
        if token_id != -100
    ]

    assert trained_token_ids, (
        "No response tokens contribute to training loss."
    )

    print(
        "\nTokens contributing to loss:",
        len(trained_token_ids),
    )

    print(
        "Text contributing to loss:",
        tokenizer.decode(
            trained_token_ids,
            skip_special_tokens=False,
        )[:1000],
    )

    test_batch = completion_collator(
        [
            tokenized_train_dataset[0],
            tokenized_train_dataset[
                min(1, len(tokenized_train_dataset) - 1)
            ],
        ]
    )

    assert (
        test_batch["input_ids"].shape
        == test_batch["attention_mask"].shape
        == test_batch["labels"].shape
    ), "The collated input, mask, and label shapes do not match."

    print(
        "Validated batch shape:",
        tuple(test_batch["input_ids"].shape),
    )

    # ---------------------------------------------------------
    # Trainer configuration for TRL 0.12.1
    # ---------------------------------------------------------

    use_bf16 = compute_dtype == torch.bfloat16
    use_fp16 = compute_dtype == torch.float16

    training_config = SFTConfig(
        output_dir=str(output_dir),

        num_train_epochs=epochs,
        learning_rate=learning_rate,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=evaluation_batch_size,
        gradient_accumulation_steps=(
            gradient_accumulation_steps
        ),

        max_seq_length=max_length,

        # Data is already tokenized and labels already exist.
        packing=False,
        remove_unused_columns=False,
        dataset_kwargs={
            "skip_prepare_dataset": True,
        },

        warmup_ratio=warmup_ratio,
        weight_decay=weight_decay,
        max_grad_norm=max_grad_norm,
        lr_scheduler_type=lr_scheduler_type,
        optim=optimizer,

        bf16=use_bf16,
        fp16=use_fp16,
        tf32=use_tf32,

        gradient_checkpointing=gradient_checkpointing,
        gradient_checkpointing_kwargs={
            "use_reentrant": False,
        },

        eval_strategy=evaluation_strategy,
        save_strategy=save_strategy,
        save_total_limit=save_total_limit,

        logging_strategy="steps",
        logging_steps=logging_steps,
        logging_first_step=True,
        disable_tqdm=disable_tqdm,

        dataloader_num_workers=0,

        seed=seed,
        data_seed=seed,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=training_config,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_evaluation_dataset,
        data_collator=completion_collator,
        processing_class=tokenizer,
    )

    # ---------------------------------------------------------
    # Evaluate before training
    # ---------------------------------------------------------

    print("\nEvaluating before adapter training...")

    initial_evaluation = trainer.evaluate(
        metric_key_prefix="before_training"
    )

    initial_loss = initial_evaluation.get(
        "before_training_loss"
    )

    initial_perplexity = (
        math.exp(initial_loss)
        if initial_loss is not None and initial_loss < 100
        else float("inf")
    )

    print(f"Initial evaluation loss:       {initial_loss}")
    print(f"Initial evaluation perplexity: {initial_perplexity}")

    trainer.save_metrics(
        "before_training",
        initial_evaluation,
    )

    # ---------------------------------------------------------
    # Train
    # ---------------------------------------------------------

    print("\nStarting QLoRA training...")

    training_result = trainer.train(
        resume_from_checkpoint=resume_from_checkpoint
    )

    trainer.log_metrics(
        "train",
        training_result.metrics,
    )
    trainer.save_metrics(
        "train",
        training_result.metrics,
    )
    trainer.save_state()

    # ---------------------------------------------------------
    # Evaluate after training
    # ---------------------------------------------------------

    print("\nEvaluating trained adapter...")

    final_evaluation = trainer.evaluate(
        metric_key_prefix="after_training"
    )

    final_loss = final_evaluation.get(
        "after_training_loss"
    )

    final_perplexity = (
        math.exp(final_loss)
        if final_loss is not None and final_loss < 100
        else float("inf")
    )

    print(f"Final evaluation loss:         {final_loss}")
    print(f"Final evaluation perplexity:   {final_perplexity}")

    if initial_loss is not None and final_loss is not None:
        loss_change = final_loss - initial_loss
        perplexity_change = (
            final_perplexity - initial_perplexity
        )

        print(f"Evaluation loss change:        {loss_change:+.6f}")
        print(
            "Evaluation perplexity change:  "
            f"{perplexity_change:+.6f}"
        )

        if final_loss < initial_loss:
            print(
                "Verdict: evaluation loss improved after training."
            )
        elif final_loss > initial_loss:
            print(
                "Verdict: evaluation loss degraded after training."
            )
        else:
            print("Verdict: evaluation loss did not change.")

    trainer.log_metrics(
        "after_training",
        final_evaluation,
    )
    trainer.save_metrics(
        "after_training",
        final_evaluation,
    )

    # ---------------------------------------------------------
    # Save final adapter
    # ---------------------------------------------------------

    final_adapter_dir = output_dir / "final_adapter"

    trainer.save_model(str(final_adapter_dir))
    tokenizer.save_pretrained(str(final_adapter_dir))

    print("\nTraining complete.")
    print(f"Final adapter saved to: {final_adapter_dir}")

    return {
        "trainer": trainer,
        "model": model,
        "tokenizer": tokenizer,
        "initial_evaluation": initial_evaluation,
        "initial_perplexity": initial_perplexity,
        "training_metrics": training_result.metrics,
        "final_evaluation": final_evaluation,
        "final_perplexity": final_perplexity,
        "adapter_directory": str(final_adapter_dir),
    }


In [19]:
os.environ["CACHE_DIR"] = "/home/jovyan/llmgenai/hf_cache"
result = run_qlora_finetuning(
    model_name="output/Qwen2.5-1.5B/v1/final_model",
    train_file="data/instructions/train.jsonl",
    evaluation_file="data/instructions/evaluation.jsonl",
    output_dir="output/Qwen2.5-1.5B/adapter-r8-all-linear",

    lora_rank=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules="all-linear",

    epochs=3,
    learning_rate=1e-4,
    train_batch_size=1,
    evaluation_batch_size=1,
    gradient_accumulation_steps=8,
    max_length=1024,

    seed=42,
)

QLoRA TRAINING CONFIGURATION
Model/CPT checkpoint:         output/Qwen2.5-1.5B/v1/final_model
Training data:                /home/jovyan/llmassign/1a/data/instructions/train.jsonl
Evaluation data:              /home/jovyan/llmassign/1a/data/instructions/evaluation.jsonl
Output directory:             /home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/adapter-r8-all-linear
GPU:                          NVIDIA A100-SXM4-80GB
Compute dtype:                torch.bfloat16
TF32 enabled:                 True
LoRA rank:                    8
LoRA alpha:                   16
LoRA scaling:                 2.00
LoRA dropout:                 0.05
Target modules:               all-linear
Epochs:                       3
Learning rate:                0.0001
Micro-batch size:             1
Gradient accumulation:        8
Effective batch size:          8
Maximum sequence length:      1024
EOS token: '<|endoftext|>' (ID: 151643)
PAD token: '<|endoftext|>' (ID: 151643)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945
Training examples:            160
Evaluation examples:          40


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]


Formatted training example:
### Instruction:
How is MAC Filtering to Enhance Port Security used?

### Response:
MAC filtering enhances port security by limiting the number of MAC addresses that can be learned within a VLAN and therefore limit the traffic in a VXLAN. Limiting the number of MAC addresses protects the switch from flooding the Ethernet switching table. Flooding of the Ethernet switching table occurs when the number of new MAC addresses that are learned causes the table to overflow, and previously learned MAC addresses are flushed from the table. The switch relearns the MAC addresses, which can impact performance and introduce security vulnerabilities. In this blueprint, MAC filtering limits the number of accepted packets that are sent to ingress-facing access interfaces based on MAC addresses. For more information about how MAC filtering works, see the MAC limiting information in Understanding MAC Limiting and MAC Move Limiting.<|endoftext|>


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]


Tokens contributing to loss: 150
Text contributing to loss: MAC filtering enhances port security by limiting the number of MAC addresses that can be learned within a VLAN and therefore limit the traffic in a VXLAN. Limiting the number of MAC addresses protects the switch from flooding the Ethernet switching table. Flooding of the Ethernet switching table occurs when the number of new MAC addresses that are learned causes the table to overflow, and previously learned MAC addresses are flushed from the table. The switch relearns the MAC addresses, which can impact performance and introduce security vulnerabilities. In this blueprint, MAC filtering limits the number of accepted packets that are sent to ingress-facing access interfaces based on MAC addresses. For more information about how MAC filtering works, see the MAC limiting information in Understanding MAC Limiting and MAC Move Limiting.<|endoftext|>
Validated batch shape: (2, 167)

Evaluating before adapter training...
{'before_tr

In [22]:
%pwd


'/home/jovyan/llmassign/1a'

In [24]:
import sys
import os

# 1. Add the 'code' folder to Python's searchable paths
sys.path.append(os.path.abspath('code'))
from print_lora import print_qlora_metrics

print_qlora_metrics("output/Qwen2.5-1.5B/adapter-r8-all-linear")



#####################################################################################################################
QLoRA RUN REPORT
#####################################################################################################################
Run directory: /home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/adapter-r8-all-linear

SAVED ARTIFACTS
final_adapter/adapter_config.json      | Present (0.00 MB)
final_adapter/training_args.bin        | Present (0.01 MB)
trainer_state.json                     | Present (0.00 MB)
before_training_results.json           | Present (0.00 MB)
train_results.json                     | Present (0.00 MB)
after_training_results.json            | Present (0.00 MB)
final_adapter/adapter_model.safetensor | Present (35.27 MB)
s                                      | 

ADAPTER CONFIGURATION
Base model/CPT checkpoint              | output/Qwen2.5-1.5B/v1/final_model
PEFT type                              | LORA
Task type                              | CAUS

In [32]:
# Adapter A — rerun because the previous run used all-linear
result_a = run_qlora_finetuning_with_chat_template(
    model_name="output/Qwen2.5-1.5B/v1/final_model",
    train_file="data/instructions/train.jsonl",
    evaluation_file="data/instructions/evaluation.jsonl",
    output_dir="output/Qwen2.5-1.5B/adapter-a-r8-q-v",
    lora_rank=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
)

print_qlora_metrics("output/Qwen2.5-1.5B/adapter-a-r8-q-v")


QLoRA TRAINING CONFIGURATION
Model/CPT checkpoint:         output/Qwen2.5-1.5B/v1/final_model
Training data:                /home/jovyan/llmassign/1a/data/instructions/train.jsonl
Evaluation data:              /home/jovyan/llmassign/1a/data/instructions/evaluation.jsonl
Output directory:             /home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/adapter-a-r8-q-v
GPU:                          NVIDIA A100-SXM4-80GB
Compute dtype:                torch.bfloat16
TF32 enabled:                 True
LoRA rank:                    8
LoRA alpha:                   16
LoRA scaling:                 2.00
LoRA dropout:                 0.05
Target modules:               ['q_proj', 'v_proj']
Epochs:                       3
Learning rate:                0.0001
Micro-batch size:             1
Gradient accumulation:        8
Effective batch size:          8
Maximum sequence length:      1024
Progress bar disabled:        True
EOS token: '<|endoftext|>' (ID: 151643)
PAD token: '<|endoftext|>' (ID: 151643)
Ch

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705
Training examples:            160
Evaluation examples:          40


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]


Formatted training example:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
How is MAC Filtering to Enhance Port Security used?<|im_end|>
<|im_start|>assistant
MAC filtering enhances port security by limiting the number of MAC addresses that can be learned within a VLAN and therefore limit the traffic in a VXLAN. Limiting the number of MAC addresses protects the switch from flooding the Ethernet switching table. Flooding of the Ethernet switching table occurs when the number of new MAC addresses that are learned causes the table to overflow, and previously learned MAC addresses are flushed from the table. The switch relearns the MAC addresses, which can impact performance and introduce security vulnerabilities. In this blueprint, MAC filtering limits the number of accepted packets that are sent to ingress-facing access interfaces based on MAC addresses. For more information about how MAC filtering works, see the MAC limiting information in Understanding MAC 

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]


Tokens contributing to loss: 151
Text contributing to loss: MAC filtering enhances port security by limiting the number of MAC addresses that can be learned within a VLAN and therefore limit the traffic in a VXLAN. Limiting the number of MAC addresses protects the switch from flooding the Ethernet switching table. Flooding of the Ethernet switching table occurs when the number of new MAC addresses that are learned causes the table to overflow, and previously learned MAC addresses are flushed from the table. The switch relearns the MAC addresses, which can impact performance and introduce security vulnerabilities. In this blueprint, MAC filtering limits the number of accepted packets that are sent to ingress-facing access interfaces based on MAC addresses. For more information about how MAC filtering works, see the MAC limiting information in Understanding MAC Limiting and MAC Move Limiting.<|im_end|>

Validated batch shape: (2, 181)

Evaluating before adapter training...
{'before_trai

In [33]:
# Adapter B — balanced
result_b = run_qlora_finetuning_with_chat_template(
    model_name="output/Qwen2.5-1.5B/v1/final_model",
    train_file="data/instructions/train.jsonl",
    evaluation_file="data/instructions/evaluation.jsonl",
    output_dir="output/Qwen2.5-1.5B/qlora-adapter-b-r16-q-v",
    lora_rank=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
)



QLoRA TRAINING CONFIGURATION
Model/CPT checkpoint:         output/Qwen2.5-1.5B/v1/final_model
Training data:                /home/jovyan/llmassign/1a/data/instructions/train.jsonl
Evaluation data:              /home/jovyan/llmassign/1a/data/instructions/evaluation.jsonl
Output directory:             /home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/qlora-adapter-b-r16-q-v
GPU:                          NVIDIA A100-SXM4-80GB
Compute dtype:                torch.bfloat16
TF32 enabled:                 True
LoRA rank:                    16
LoRA alpha:                   32
LoRA scaling:                 2.00
LoRA dropout:                 0.05
Target modules:               ['q_proj', 'v_proj']
Epochs:                       3
Learning rate:                0.0001
Micro-batch size:             1
Gradient accumulation:        8
Effective batch size:          8
Maximum sequence length:      1024
Progress bar disabled:        True
EOS token: '<|endoftext|>' (ID: 151643)
PAD token: '<|endoftext|>' (ID: 15

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410
Training examples:            160
Evaluation examples:          40


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]


Formatted training example:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
How is MAC Filtering to Enhance Port Security used?<|im_end|>
<|im_start|>assistant
MAC filtering enhances port security by limiting the number of MAC addresses that can be learned within a VLAN and therefore limit the traffic in a VXLAN. Limiting the number of MAC addresses protects the switch from flooding the Ethernet switching table. Flooding of the Ethernet switching table occurs when the number of new MAC addresses that are learned causes the table to overflow, and previously learned MAC addresses are flushed from the table. The switch relearns the MAC addresses, which can impact performance and introduce security vulnerabilities. In this blueprint, MAC filtering limits the number of accepted packets that are sent to ingress-facing access interfaces based on MAC addresses. For more information about how MAC filtering works, see the MAC limiting information in Understanding MAC 

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]


Tokens contributing to loss: 151
Text contributing to loss: MAC filtering enhances port security by limiting the number of MAC addresses that can be learned within a VLAN and therefore limit the traffic in a VXLAN. Limiting the number of MAC addresses protects the switch from flooding the Ethernet switching table. Flooding of the Ethernet switching table occurs when the number of new MAC addresses that are learned causes the table to overflow, and previously learned MAC addresses are flushed from the table. The switch relearns the MAC addresses, which can impact performance and introduce security vulnerabilities. In this blueprint, MAC filtering limits the number of accepted packets that are sent to ingress-facing access interfaces based on MAC addresses. For more information about how MAC filtering works, see the MAC limiting information in Understanding MAC Limiting and MAC Move Limiting.<|im_end|>

Validated batch shape: (2, 181)

Evaluating before adapter training...
{'before_trai

In [27]:
print_qlora_metrics("output/Qwen2.5-1.5B/qlora-adapter-b-r16-q-v")


#####################################################################################################################
QLoRA RUN REPORT
#####################################################################################################################
Run directory: /home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/qlora-adapter-b-r16-q-v

SAVED ARTIFACTS
final_adapter/adapter_config.json      | Present (0.00 MB)
final_adapter/training_args.bin        | Present (0.01 MB)
trainer_state.json                     | Present (0.00 MB)
before_training_results.json           | Present (0.00 MB)
train_results.json                     | Present (0.00 MB)
after_training_results.json            | Present (0.00 MB)
final_adapter/adapter_model.safetensor | Present (8.33 MB)
s                                      | 

ADAPTER CONFIGURATION
Base model/CPT checkpoint              | output/Qwen2.5-1.5B/v1/final_model
PEFT type                              | LORA
Task type                              | CAU

In [5]:
# Adapter C — high capacity
result_c = run_qlora_finetuning_with_chat_template(
    model_name="output/Qwen2.5-1.5B/v1/final_model",
    train_file="data/instructions/train.jsonl",
    evaluation_file="data/instructions/evaluation.jsonl",
    output_dir="output/Qwen2.5-1.5B/qlora-adapter-c-r32-q-v-o",
    lora_rank=32,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "o_proj"],
)

print_qlora_metrics("output/Qwen2.5-1.5B/qlora-adapter-c-r32-q-v-o")

QLoRA TRAINING CONFIGURATION
Model/CPT checkpoint:         output/Qwen2.5-1.5B/v1/final_model
Training data:                /home/jovyan/llmassign/1a/data/instructions/train.jsonl
Evaluation data:              /home/jovyan/llmassign/1a/data/instructions/evaluation.jsonl
Output directory:             /home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/qlora-adapter-c-r32-q-v-o
GPU:                          NVIDIA A100-SXM4-80GB
Compute dtype:                torch.bfloat16
TF32 enabled:                 True
LoRA rank:                    32
LoRA alpha:                   32
LoRA scaling:                 1.00
LoRA dropout:                 0.05
Target modules:               ['q_proj', 'v_proj', 'o_proj']
Epochs:                       3
Learning rate:                0.0001
Micro-batch size:             1
Gradient accumulation:        8
Effective batch size:          8
Maximum sequence length:      1024
Progress bar disabled:        True
EOS token: '<|endoftext|>' (ID: 151643)
PAD token: '<|endoftex

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 7,110,656 || all params: 1,550,824,960 || trainable%: 0.4585


Parameter 'function'=<function run_qlora_finetuning_with_chat_template.<locals>.format_record at 0x7ed5041189a0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Training examples:            160
Evaluation examples:          40


...


Formatted training example:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
How is MAC Filtering to Enhance Port Security used?<|im_end|>
<|im_start|>assistant
MAC filtering enhances port security by limiting the number of MAC addresses that can be learned within a VLAN and therefore limit the traffic in a VXLAN. Limiting the number of MAC addresses protects the switch from flooding the Ethernet switching table. Flooding of the Ethernet switching table occurs when the number of new MAC addresses that are learned causes the table to overflow, and previously learned MAC addresses are flushed from the table. The switch relearns the MAC addresses, which can impact performance and introduce security vulnerabilities. In this blueprint, MAC filtering limits the number of accepted packets that are sent to ingress-facing access interfaces based on MAC addresses. For more information about how MAC filtering works, see the MAC limiting information in Understanding MAC 

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]


Tokens contributing to loss: 151
Text contributing to loss: MAC filtering enhances port security by limiting the number of MAC addresses that can be learned within a VLAN and therefore limit the traffic in a VXLAN. Limiting the number of MAC addresses protects the switch from flooding the Ethernet switching table. Flooding of the Ethernet switching table occurs when the number of new MAC addresses that are learned causes the table to overflow, and previously learned MAC addresses are flushed from the table. The switch relearns the MAC addresses, which can impact performance and introduce security vulnerabilities. In this blueprint, MAC filtering limits the number of accepted packets that are sent to ingress-facing access interfaces based on MAC addresses. For more information about how MAC filtering works, see the MAC limiting information in Understanding MAC Limiting and MAC Move Limiting.<|im_end|>

Validated batch shape: (2, 181)

Evaluating before adapter training...
{'before_trai

NameError: name 'print_qlora_metrics' is not defined

In [6]:
import sys
import os

# 1. Add the 'code' folder to Python's searchable paths
sys.path.append(os.path.abspath('code'))
from print_lora import print_qlora_metrics

print_qlora_metrics("output/Qwen2.5-1.5B/qlora-adapter-c-r32-q-v-o")


#####################################################################################################################
QLoRA RUN REPORT
#####################################################################################################################
Run directory: /home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/qlora-adapter-c-r32-q-v-o

SAVED ARTIFACTS
final_adapter/adapter_config.json      | Present (0.00 MB)
final_adapter/training_args.bin        | Present (0.01 MB)
trainer_state.json                     | Present (0.00 MB)
before_training_results.json           | Present (0.00 MB)
train_results.json                     | Present (0.00 MB)
after_training_results.json            | Present (0.00 MB)
final_adapter/adapter_model.safetensor | Present (27.15 MB)
s                                      | 

ADAPTER CONFIGURATION
Base model/CPT checkpoint              | output/Qwen2.5-1.5B/v1/final_model
PEFT type                              | LORA
Task type                              | 

In [3]:
import sys
import os
sys.path.append(os.path.abspath('code'))
from compare_qlora_adapters import print_adapter_comparison


In [6]:
print_adapter_comparison("/home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/")


QLoRA ADAPTER CONFIGURATION COMPARISON
Parent folder: /home/jovyan/llmassign/1a/output/Qwen2.5-1.5B

+======================+===============+========+=========+===========+================================+
| Run                  | Assignment    | Rank   | Alpha   | Dropout   | Target modules                 |
+======================+===============+========+=========+===========+================================+
| adapter-a-r8-q-v     | A / Low       | 8      | 16      | 0.05      | q_proj, v_proj                 |
+----------------------+---------------+--------+---------+-----------+--------------------------------+
| adapter-b-r16-q-v    | B / Balanced  | 16     | 32      | 0.05      | q_proj, v_proj                 |
+----------------------+---------------+--------+---------+-----------+--------------------------------+
| adapter-c-r32-q-v-o  | C / High      | 32     | 32      | 0.05      | o_proj, q_proj, v_proj         |
+----------------------+---------------+--------+---------

[{'directory': 'adapter-a-r8-q-v',
  'capacity': 'A / Low',
  'base_model': 'output/Qwen2.5-1.5B/v1/final_model',
  'rank': 8,
  'alpha': 16,
  'dropout': 0.05,
  'targets': 'q_proj, v_proj',
  'initial_loss': 2.3801920413970947,
  'final_loss': 2.2697436809539795,
  'initial_ppl': 10.806978051827334,
  'final_ppl': 9.676920117228633,
  'ppl_change_percent': -10.45674312633236,
  'train_loss': 2.3173876206080117,
  'runtime': 169.8362,
  'epochs': 3.0,
  'best_epoch': 3.0,
  'best_eval_loss': 2.2697436809539795},
 {'directory': 'adapter-b-r16-q-v',
  'capacity': 'B / Balanced',
  'base_model': 'output/Qwen2.5-1.5B/v1/final_model',
  'rank': 16,
  'alpha': 32,
  'dropout': 0.05,
  'targets': 'q_proj, v_proj',
  'initial_loss': 2.3801920413970947,
  'final_loss': 2.239147663116455,
  'initial_ppl': 10.806978051827334,
  'final_ppl': 9.385328415782975,
  'ppl_change_percent': -13.154922950953662,
  'train_loss': 2.2899423360824587,
  'runtime': 169.4984,
  'epochs': 3.0,
  'best_epoch': 3